# Clean benchmark on grouping v2 folds: seeds 43 and 44 (YOLO11s, YOLOv8s)

Notebook 10 gave one seed (42) per detector on the v2 folds. The detector ranking flipped between v1 and v2 and a single seed cannot tell a real gap from noise. This notebook adds seeds 43 and 44 on the same v2 folds (`data_splits_v2/clean5`) and the same fixed protocol (100 epochs, no early stopping, last.pt), giving 3 seeds x 5 folds per detector.
* **PART = "A"**: YOLO11s, seeds 43 and 44, 5 folds each = 10 runs (about 2.5 h on T4 x2)
* **PART = "B"**: YOLOv8s, seeds 43 and 44, 5 folds each = 10 runs (about 2.5 h on T4 x2)
Run A and B as two separate Save Versions (e.g. on two accounts). Settings: GPU T4 x2, Internet ON. Start with SMOKE = True (2 epochs, 1 run).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
PART = "A"                     # "A" = YOLO11s, "B" = YOLOv8s
MODEL = {"A": "yolo11s.pt", "B": "yolov8s.pt"}[PART]
SEEDS = [43, 44]
FOLDS = [0, 1, 2, 3, 4]
SPLITS = f"{REPO}/data_splits_v2"
assert os.path.isdir(f"{SPLITS}/clean5"), "data_splits_v2 missing: push it and re-run the clone cell"
if SMOKE:
    EPOCHS, OUT = 2, f"/kaggle/working/results/v2s_{PART}_smoke"; FOLDS = [0]; SEEDS = [43]
else:
    EPOCHS, OUT = 100, f"/kaggle/working/results/v2s_{PART}"

def make_cmd(model, fold, seed):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", "clean", "--fold", str(fold), "--seed", str(seed),
                "--model", model, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
                "--splits-dir", SPLITS, "--out", OUT]
    return f
TASKS = [(f"v2sclean_f{f}_s{s}_{MODEL[:-3]}", make_cmd(MODEL, f, s), None) for s in SEEDS for f in FOLDS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        while needs and not os.path.exists(needs):          # e.g. arm 6 waits for the FracAtlas pre-trained checkpoint
            if any(n.startswith("fa_pre_") for n in failed):
                break
            time.sleep(20)
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import shutil
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_benchmark.py", "--runs", OUT, "--out", f"{OUT}/benchmark_v2s_part{PART}"])
shutil.make_archive(f"/kaggle/working/v2s_{PART}_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `v2s_<PART>_results.zip`, unzip once into `results/v2s_<PART>/` (rename the unzipped folder), commit and push; then `python scripts/evaluation/compare_v2_seeds.py` (needs both parts).